<a href="https://colab.research.google.com/github/marcocintra/Atmosphere/blob/master/SEARCH_FOR_CASE_STUDY_NAGOYA_MAPS_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zip files adjusts

In [ ]:
#Download Nagoya_TEC_maps_intersection_raw_files_from_Sept_20_2024_to_Sept_end_case_study.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
%pwd

In [ ]:
!ls

In [ ]:
!unzip -q 'Nagoya_TEC_maps_intersection_raw_files_from_Sept_20_2024_to_Sept_end_case_study.zip'

# Data files adjusts

In [ ]:
!ls

In [ ]:
%cd '/content/Nagoya_TEC_maps_intersection_raw_files_from_Sept_20_2024_to_Sept_end_case_study'

In [ ]:
import os
import glob
from collections import defaultdict

base_path = './'

file_pattern = os.path.join(base_path, '**', '2024*.nc')
file_list = sorted(glob.glob(file_pattern, recursive=True))

file_count_by_day = defaultdict(int)

for file_name in file_list:

    day_dir = os.path.basename(os.path.dirname(file_name))
    file_count_by_day[day_dir] += 1

for day, count in sorted(file_count_by_day.items()):
    if count < 144:
        print(f"Day {day}: {count} files")

In [ ]:
import numpy as np
import xarray as xr
import os
import re
import glob

file_list = glob.glob("./2024*.nc")

pattern = re.compile(r'(\d{8})(\d{2})_atec')

def extract_datetime(file_name):
    match = pattern.search(file_name)
    if match:
        date = match.group(1)
        hour = match.group(2)
        return date, int(hour)
    return None, None

files_by_date = {}

for file_name in file_list:
    date, hour = extract_datetime(file_name)
    if date:
        if date not in files_by_date:
            files_by_date[date] = []
        files_by_date[date].append((file_name, hour))

sorted_dates = sorted(files_by_date.keys())
print(f"\nDates to be processed in chronological order: {sorted_dates}")

print("\nProcessing data by day (in chronological order)...")
for date in sorted_dates:
    file_hour_pairs = files_by_date[date]

    file_hour_pairs.sort(key=lambda x: x[1])

    print(f"Processing the day: {date}, {len(file_hour_pairs)} files found")

    final_data = np.zeros((288, 242, 221))

    for file_name, hour in file_hour_pairs:
        try:
            print(f"  Processing: {file_name}, Hour: {hour}")

            ds_disk = xr.open_dataset(file_name)
            tec = ds_disk['atec'][19:261, 140:361].values

            for timestep in range(12):

                global_idx = hour * 12 + timestep

                timestep_slice = tec[:, :, timestep]

                final_data[global_idx] = timestep_slice

            ds_disk.close()

        except Exception as e:
            print(f"Error processing file {file_name}: {e}")

    print(f"Final array shape: {final_data.shape}")

    output_file = f"atec{date}.npy"
    np.save(output_file, final_data)

    print(f"File {output_file} saved successfully! Shape: {final_data.shape}")

print("\nProcessing completed!")

In [ ]:
!ls -lR ./*.npy | wc -l

In [ ]:
import numpy as np
import glob
import re
from datetime import datetime

npy_files = glob.glob("atec2024*.npy")

def extract_doy(filename):
    match = re.search(r'atec(\d{8})\.npy', filename)
    if match:
        date_str = match.group(1)
        try:
            date_obj = datetime.strptime(date_str, '%Y%m%d')
            return date_obj.timetuple().tm_yday
        except ValueError:
            return 0
    return 0

npy_files.sort(key=extract_doy)

print(f"Found {len(npy_files)} NPY files to process.")

all_data = []

for i, file in enumerate(npy_files):
    doy = extract_doy(file)
    print(f"Processing {file} (DOY {doy}) - {i+1}/{len(npy_files)}")

    data = np.load(file)

    all_data.append(data)

    print(f"Shape: {data.shape}")

combined_data = np.concatenate(all_data, axis=0)

print(f"\nCombined data. Final shape: {combined_data.shape}")

output_file = "Nagoya_TEC_maps_intersection_from_Sept_20_2024_to_Sept_end_case_study.npy"
np.save(output_file, combined_data)

print(f"\nFile {output_file} saved successfully!")
print(f"File size: {combined_data.nbytes / (1024**2):.2f} MB")

In [ ]:
!ls -lh *.npy

In [ ]:
from google.colab import files

In [ ]:
%pwd

In [ ]:
from google.colab import files

In [ ]:
!ls -lh Nagoya*.npy

# Dates adjusts

In [ ]:
import numpy as np

In [ ]:
nagoya_2024 = np.load('Nagoya_TEC_maps_intersection_raw_files_from_Sept_20_2024_to_Sept_end_case_study.npy')

In [ ]:
import numpy as np
from datetime import datetime, timedelta

def doy_to_date(year, doy):

    return datetime(year, 1, 1) + timedelta(days=doy - 1)

doys = {

    2024: {
        9: [264, 265, 266, 267, 268, 269, 270, 271, 272, 274],
        }
}

datetime_list = []

for year in doys.keys():
    for month in doys[year].keys():
        for doy in doys[year][month]:
            base_date = doy_to_date(year, doy)
            for hour in range(24):
                for minute in range(0, 60, 5):
                    dt = datetime(base_date.year, base_date.month, base_date.day, hour, minute, 0)
                    datetime_list.append(dt)

nagoya_datetimes_2024 = np.array(datetime_list, dtype='datetime64[s]')

print(f"Total number of datetime points: {len(nagoya_datetimes_2024)}")
print(f"First datetime: {nagoya_datetimes_2024[0]}")
print(f"Last datetime: {nagoya_datetimes_2024[-1]}")

print("\nSample of first 10 datetime entries:")
for dt in nagoya_datetimes_2024[:10]:
    print(dt)

print("\nSample of last 10 datetime entries:")
for dt in nagoya_datetimes_2024[-10:]:
    print(dt)

In [ ]:
nagoya_datetimes_2024.shape

In [ ]:
nagoya_datetimes_2024

In [ ]:
nagoya_2024.shape

In [ ]:
print(len(nagoya_2024.tolist()))
print(len(nagoya_2024[0].tolist()))
print(len(nagoya_2024[0][0].tolist()))

In [ ]:
len(nagoya_2024[0:12])

In [ ]:
nagoya_2024_shaped = np.reshape(nagoya_2024, (-1, 242, 221))

In [ ]:
np.shape(nagoya_2024_shaped)

In [ ]:
np.shape(nagoya_2024_shaped.tolist())

## Create dataframe

In [ ]:
nagoya_datetimes_2024.shape

In [ ]:
import pandas as pd

data = {
    'DATETIME': nagoya_datetimes_2024,
    'TECMAP': nagoya_2024_shaped.tolist()
}

df_mapas_nagoya_2024 = pd.DataFrame(data)

df_mapas_nagoya_2024

In [ ]:
np.array(df_mapas_nagoya_2024.iloc[0]['TECMAP'])

In [ ]:
np.array(df_mapas_nagoya_2024.iloc[0]['TECMAP']).shape

In [ ]:
df_mapas_nagoya_2024.to_pickle("./TF_Nagoya_TEC_maps_from_Sept_05_2024_to_Dec_2024_case_study.pkl")

In [ ]:
mapas_nagoya_2024 = np.array(df_mapas_nagoya_2024.iloc[:]['TECMAP'])

In [ ]:
np.shape(df_mapas_nagoya_2024)

In [ ]:
np_mapas_nagoya_2024 = []
for i in range(len(mapas_nagoya_2024)):
    np_mapas_nagoya_2024.append(mapas_nagoya_2024[i])
np_mapas_nagoya_2024 = np.array(np_mapas_nagoya_2024)

In [ ]:
np.shape(np_mapas_nagoya_2024)

In [ ]:
type(np_mapas_nagoya_2024)

In [ ]:
np.save('TF_Nagoya_TEC_maps_from_Sept_05_2024_to_Dec_2024_case_study.npy', np_mapas_nagoya_2024)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -lh TF*

In [ ]:
%cp /content/TF_Nagoya_TEC_maps_from_Sept_05_2024_to_Dec_2024_case_study.pkl /content/drive/MyDrive/TF_Nagoya_TEC_maps_from_Sept_05_2024_to_Dec_2024_case_study.pkl

In [ ]:
%cp /content/TF_Nagoya_TEC_maps_from_Sept_05_2024_to_Dec_2024_case_study.npy /content/drive/MyDrive/TF_Nagoya_TEC_maps_from_Sept_05_2024_to_Dec_2024_case_study.npy